<a href="https://colab.research.google.com/github/minbj1226/pytorch-basics/blob/main/06_Model_Save_Load.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. state_dict란?
- 모델의 학습 가능한 파라미터와 버퍼 값을 딕셔너리 형태로 저장한 것

### 2. torch.save()는 무엇을 할 때 사용하는가?
- 모델의 가중치, checkpoint, 텐서 등의 객체를 파일로 저장할 때 사용

### 3. 모델 가중치는 저장 방법
- model.state_dict()를 torch.save()로 저장

### 4. 저장한 모델은 불러오는 방법
- 같은 구조의 모델을 만든 뒤 torch.load()로 불러온 값을 model.load_state_dict()에 넣어 복원

### 5. 불러온 뒤 model.eval() 해야 하는 이유
- 추론 또는 평가 시 Dropout과 BatchNorm이 평가 방식으로 동작하기 위해 model.eval()을 사용

### 6. checkpoint 저장이 필요한 이유
- 학습 중간 상태를 저장해 두었다가, 중단된 지점부터 다시 학습을 이어가기 위해 checkpoint를 저장한다.

### 3가지 핵심
1. ```torch.save```
2. ```torch.load```
3. ```torch.nn.Module.load_state_dict```

In [10]:
import torch.nn as nn
import torch.functional as F
import torch.optim as optim

class TheModelClass(nn.Module):
  def __init__(self):
    super(TheModelClass, self).__init__()
    self.conv1 = nn.Conv2d(3, 6, 5)
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(6, 16, 5)
    self.fc1 = nn.Linear(16 * 5 * 5, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.view(-1, 16 * 5 * 5)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

model = TheModelClass()

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

Model's state_dict:
conv1.weight 	 torch.Size([6, 3, 5, 5])
conv1.bias 	 torch.Size([6])
conv2.weight 	 torch.Size([16, 6, 5, 5])
conv2.bias 	 torch.Size([16])
fc1.weight 	 torch.Size([120, 400])
fc1.bias 	 torch.Size([120])
fc2.weight 	 torch.Size([84, 120])
fc2.bias 	 torch.Size([84])
fc3.weight 	 torch.Size([10, 84])
fc3.bias 	 torch.Size([10])
Optimizer's state_dict:
state 	 {}
param_groups 	 [{'lr': 0.001, 'momentum': 0.9, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}]


In [13]:
import torch

PATH = "model.pth"

torch.save(model.state_dict(), PATH)

In [15]:
model = TheModelClass()
model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()

TheModelClass(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)